# Tension Board 2 Mirror: Feature Engineering

The goal of this notebook is to convert raw climb descriptions into a clean modelling table. Each row of the final table corresponds to a single climb-angle observation, and each column is a numeric feature that may help predict grade.

## Modelling idea

A climb's grade should depend on more than just angle. It should also depend on the geometry and sequencing of the holds used. To capture that, this notebook builds features from three sources:

1. **Wall configuration**  
   Examples: angle, board geometry, mirrored placements.

2. **Route structure**  
   Examples: number of holds, spatial spread, height gained, move lengths, left/right balance, and other frame-derived quantities.

When this was initially done, we added:

3. **Hold difficulty priors** 

However, that makes it quite circular -- we'd be using the difficulty data to create difficulty scores to then predict difficulty data. The difficulty is already baked in there, so it is not a very good independent model. Heuristically, I don't think this is a big deal if we **just** want to predict V-grades, but we'll leave it out of our analysis in order to see what sorts of features actually help determine the difficulty of a climb. We'll add it back in notebook 07 and create a *leaky model*. 

## Output

The final product is a saved feature matrix that is reused in the predictive modelling and deep learning notebooks.

## Notebook Structure

1. [Setup and Imports](#setup-and-imports)
2. [Feature Extraction](#feature-extraction)
3. [Visualizing Key Features](#visualizing-key-features)
4. [Conclusion](#conclusion)

# Setup and Imports

This section loads the database, auxiliary tables, and the hold-difficulty table produced in notebook 03.


In [ ]:
"""
==================================
Setup and Imports
==================================
"""

# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.patches as mpatches

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform

import sqlite3

import re
import os
from collections import defaultdict

import ast

from PIL import Image

# Set some display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style
palette=['steelblue', 'coral', 'seagreen']  #(for multi-bar graphs)

# Set board image for some visual analysis
board_img = Image.open('../images/tb2_board_12x12_composite.png')

# Connect to the database
DB_PATH="../data/tb2.db"
conn = sqlite3.connect(DB_PATH)

# Create output directories
os.makedirs('../data/04_climb_features', exist_ok=True)
os.makedirs('../images/04_climb_features', exist_ok=True)


In [ ]:
"""
==================================
Query our data from the DB
==================================

This time we restrict to where `layout_id=10` for the TB2 Mirror.
We will also restrict ourselves to an angle of at most 50, since according to our grade vs angle distribution in notebook 01, things start to look a bit weird past 50.
(Probably a bias towards climbers who can actually climb that steep). We will encode this directly into our query.
"""

# Query climbs data
climbs_query = """
SELECT
    c.uuid,
    c.name AS climb_name,
    c.setter_username,
    c.layout_id AS layout_id,
    c.description,
    c.is_nomatch,
    c.is_listed,
    l.name AS layout_name,
    p.name AS board_name,
    c.frames,
    cs.angle,
    cs.display_difficulty,
    dg.boulder_name AS boulder_grade,
    cs.ascensionist_count,
    cs.quality_average,
    cs.fa_at
FROM climbs c
JOIN layouts l ON c.layout_id = l.id
JOIN products p ON l.product_id = p.id
JOIN climb_stats cs ON c.uuid = cs.climb_uuid
JOIN difficulty_grades dg ON ROUND(cs.display_difficulty) = dg.difficulty
WHERE cs.display_difficulty IS NOT NULL AND c.is_listed=1 AND c.layout_id=10 AND cs.angle <= 50
"""

# Query information about placements (and their mirrors)
placements_query = """
SELECT
    p.id AS placement_id,
    h.x,
    h.y,
    p.default_placement_role_id AS default_role_id,
    p.set_id AS set_id,
    s.name AS set_name,
    p_mirror.id AS mirror_placement_id
FROM placements p
JOIN holes h ON p.hole_id = h.id
JOIN sets s ON p.set_id = s.id
LEFT JOIN holes h_mirror ON h.mirrored_hole_id = h_mirror.id
LEFT JOIN placements p_mirror ON p_mirror.hole_id = h_mirror.id AND p_mirror.layout_id = p.layout_id
WHERE p.layout_id = 10
"""

# Load it into a DataFrame
df_climbs = pd.read_sql_query(climbs_query, conn)
df_placements = pd.read_sql_query(placements_query, conn)

# Load the hold-level difficulty table created in notebook 03
df_hold_difficulty = pd.read_csv('../data/03_hold_difficulty/hold_difficulty_scores.csv', index_col='placement_id')

In [ ]:
print("Difficulty-related columns loaded from Notebook 03:")
print([c for c in df_hold_difficulty.columns if 'difficulty' in c.lower()])

assert 'overall_difficulty' in df_hold_difficulty.columns, "Missing overall_difficulty"

In [ ]:
df_hold_difficulty

In [ ]:
placement_coords = {
    row['placement_id']: (row['x'], row['y'])
    for _, row in df_placements.iterrows()
}

board_width = 144
board_height = 144

x_min, x_max = -68, 68
y_min, y_max = 0, 144

# Role definitions (TB2)
ROLE_DEFINITIONS = {
    'start': 5,
    'middle': 6,
    'finish': 7,
    'foot': 8
}

HAND_ROLE_IDS = [5, 6, 7]
FOOT_ROLE_IDS = [8]



In [ ]:
"""
==================================
Parse Frame function
==================================
"""

def parse_frames(frames_str):
    """
    Parse frames string into list of (placement_id, role_id) tuples.
    
    Parameters:
    -----------
    frames_str : str
        Frame string like "p1r5p2r6p3r8"
    
    Returns:
    --------
    list of tuples: [(placement_id, role_id), ...]
    """
    if not isinstance(frames_str, str):
        return []
    
    matches = re.findall(r'p(\d+)r(\d+)', frames_str)
    return [(int(p), int(r)) for p, r in matches]


def get_role_type(role_id):
    """Map role_id to role type string."""
    for role_type, rid in ROLE_DEFINITIONS.items():
        if role_id == rid:
            return role_type
    return 'unknown'


# Test
test_frames = "p1r5p2r6p3r8p4r5"
parsed = parse_frames(test_frames)
print(f"Test parse: {parsed}")

# Feature Extraction

This is the core notebook section. The aim is to translate the raw `frames` string into a route-level numerical representation suitable for regression or classification models.


In [ ]:
"""
==================================
Feature Exraction Function
==================================
"""

In [ ]:
def extract_features(row, placement_coords):
    """
    Extract a trimmed set of clean geometric/spatial features.
    No hold-difficulty-derived features are used.
    """
    features = {}

    holds = parse_frames(row['frames'])
    angle = row['angle']

    if not holds:
        return None

    hold_data = []
    for placement_id, role_id in holds:
        coords = placement_coords.get(placement_id, (None, None))
        if coords[0] is None:
            continue

        role_type = get_role_type(role_id)
        is_hand = role_id in HAND_ROLE_IDS
        is_foot = role_id in FOOT_ROLE_IDS

        hold_data.append({
            'placement_id': placement_id,
            'x': coords[0],
            'y': coords[1],
            'role_type': role_type,
            'is_hand': is_hand,
            'is_foot': is_foot,
        })

    if not hold_data:
        return None

    df_holds = pd.DataFrame(hold_data)

    hand_holds = df_holds[df_holds['is_hand']]
    foot_holds = df_holds[df_holds['is_foot']]
    start_holds = df_holds[df_holds['role_type'] == 'start']
    finish_holds = df_holds[df_holds['role_type'] == 'finish']
    middle_holds = df_holds[df_holds['role_type'] == 'middle']

    xs = df_holds['x'].to_numpy()
    ys = df_holds['y'].to_numpy()

    description = row.get('description', '')
    if pd.isna(description):
        description = ''

    center_x = (x_min + x_max) / 2

    # Basic
    features['angle'] = angle
    features['angle_squared'] = angle ** 2

    features['total_holds'] = len(df_holds)
    features['hand_holds'] = len(hand_holds)
    features['foot_holds'] = len(foot_holds)
    features['start_holds'] = len(start_holds)
    features['finish_holds'] = len(finish_holds)
    features['middle_holds'] = len(middle_holds)

    features['is_nomatch'] = int(
        (row['is_nomatch'] == 1) or
        bool(re.search(r'\bno\s*match(ing)?\b', description, flags=re.IGNORECASE))
    )

    # Spatial
    features['mean_y'] = np.mean(ys)
    features['std_x'] = np.std(xs) if len(xs) > 1 else 0.0
    features['std_y'] = np.std(ys) if len(ys) > 1 else 0.0
    features['range_x'] = np.max(xs) - np.min(xs)
    features['range_y'] = np.max(ys) - np.min(ys)
    features['min_y'] = np.min(ys)
    features['max_y'] = np.max(ys)
    features['height_gained'] = features['max_y'] - features['min_y']

    # Start / finish heights
    start_height = start_holds['y'].mean() if len(start_holds) > 0 else np.nan
    finish_height = finish_holds['y'].mean() if len(finish_holds) > 0 else np.nan

    features['height_gained_start_finish'] = (
        finish_height - start_height
        if pd.notna(start_height) and pd.notna(finish_height)
        else np.nan
    )

    # Density / symmetry
    bbox_area = features['range_x'] * features['range_y']
    features['bbox_area'] = bbox_area
    features['hold_density'] = features['total_holds'] / bbox_area if bbox_area > 0 else 0.0
    features['holds_per_vertical_foot'] = features['total_holds'] / max(features['range_y'], 1)

    left_holds = (df_holds['x'] < center_x).sum()
    features['left_ratio'] = left_holds / features['total_holds'] if features['total_holds'] > 0 else 0.5
    features['symmetry_score'] = 1 - abs(features['left_ratio'] - 0.5) * 2

    y_median = np.median(ys)
    upper_holds = (df_holds['y'] > y_median).sum()
    features['upper_ratio'] = upper_holds / features['total_holds']

    # Hand reach
    if len(hand_holds) >= 2:
        hand_points = hand_holds[['x', 'y']].to_numpy()
        hand_distances = pdist(hand_points)

        hand_xs = hand_holds['x'].to_numpy()
        hand_ys = hand_holds['y'].to_numpy()

        features['mean_hand_reach'] = float(np.mean(hand_distances))
        features['max_hand_reach'] = float(np.max(hand_distances))
        features['std_hand_reach'] = float(np.std(hand_distances))
        features['hand_spread_x'] = float(hand_xs.max() - hand_xs.min())
        features['hand_spread_y'] = float(hand_ys.max() - hand_ys.min())
    else:
        features['mean_hand_reach'] = 0.0
        features['max_hand_reach'] = 0.0
        features['std_hand_reach'] = 0.0
        features['hand_spread_x'] = 0.0
        features['hand_spread_y'] = 0.0

    # Hand-foot distances
    if len(hand_holds) > 0 and len(foot_holds) > 0:
        hand_points = hand_holds[['x', 'y']].to_numpy()
        foot_points = foot_holds[['x', 'y']].to_numpy()

        dists = []
        for hx, hy in hand_points:
            for fx, fy in foot_points:
                dists.append(np.sqrt((hx - fx)**2 + (hy - fy)**2))
        dists = np.asarray(dists)

        features['min_hand_to_foot'] = float(np.min(dists))
        features['mean_hand_to_foot'] = float(np.mean(dists))
        features['std_hand_to_foot'] = float(np.std(dists))
    else:
        features['min_hand_to_foot'] = 0.0
        features['mean_hand_to_foot'] = 0.0
        features['std_hand_to_foot'] = 0.0

    # Global geometry
    points = np.column_stack([xs, ys])

    if len(df_holds) >= 3:
        try:
            hull = ConvexHull(points)
            features['convex_hull_area'] = float(hull.volume)
            features['hull_area_to_bbox_ratio'] = features['convex_hull_area'] / max(bbox_area, 1)
        except Exception:
            features['convex_hull_area'] = np.nan
            features['hull_area_to_bbox_ratio'] = np.nan
    else:
        features['convex_hull_area'] = 0.0
        features['hull_area_to_bbox_ratio'] = 0.0

    if len(df_holds) >= 2:
        pairwise = pdist(points)
        features['mean_pairwise_distance'] = float(np.mean(pairwise))
        features['std_pairwise_distance'] = float(np.std(pairwise))
    else:
        features['mean_pairwise_distance'] = 0.0
        features['std_pairwise_distance'] = 0.0

    if len(df_holds) >= 2:
        sorted_idx = np.argsort(ys)
        sorted_points = points[sorted_idx]

        path_length = 0.0
        for i in range(len(sorted_points) - 1):
            dx = sorted_points[i + 1, 0] - sorted_points[i, 0]
            dy = sorted_points[i + 1, 1] - sorted_points[i, 1]
            path_length += np.sqrt(dx**2 + dy**2)

        features['path_length_vertical'] = path_length
        features['path_efficiency'] = features['height_gained'] / max(path_length, 1)
    else:
        features['path_length_vertical'] = 0.0
        features['path_efficiency'] = 0.0

    # Normalized / relative
    features['mean_y_normalized'] = (features['mean_y'] - y_min) / board_height
    features['start_height_normalized'] = (
        (start_height - y_min) / board_height if pd.notna(start_height) else np.nan
    )
    features['finish_height_normalized'] = (
        (finish_height - y_min) / board_height if pd.notna(finish_height) else np.nan
    )
    features['mean_y_relative_to_start'] = (
        features['mean_y'] - start_height if pd.notna(start_height) else np.nan
    )
    features['spread_x_normalized'] = features['range_x'] / board_width
    features['spread_y_normalized'] = features['range_y'] / board_height

    y_q75 = np.percentile(ys, 75)
    y_q25 = np.percentile(ys, 25)
    features['y_q75'] = y_q75
    features['y_iqr'] = y_q75 - y_q25

    # Optional engineered clean feature
    features['complexity_score'] = (
        features['mean_hand_reach']
        * np.log1p(features['total_holds'])
        * (1 + features['hold_density'])
    )

    return features

## Sanity Check on One Example

Before extracting features for the entire dataset, we inspect one representative climb to confirm that the parsing logic and the computed geometric summaries behave as expected. Let's do the climb "Ooo La La" from notebook two.

![Ooo La La](../images/02_hold_stats/Ooo_La_La.png)


In [ ]:
extract_features(df_climbs.iloc[10000], placement_coords)

The printed example above is an important checkpoint. If the parsed placements, role counts, or geometric summaries look unreasonable here, then the full feature matrix will inherit those mistakes.


## Extract Features or all climbs

In [ ]:
"""
==================================
Extract features for all climbs
==================================
"""

from tqdm import tqdm # Progess bar. This will take a while.

print(f"Extracting features for {len(df_climbs)} climbs...")

feature_list = []

for idx, row in tqdm(df_climbs.iterrows(), total=len(df_climbs)):
    features = extract_features(row, placement_coords)
    if features:
        features['climb_uuid'] = row['uuid']
        features['display_difficulty'] = row['display_difficulty']
        feature_list.append(features)

df_features = pd.DataFrame(feature_list)
df_features = df_features.set_index('climb_uuid')

print(f"\nExtracted features for {len(df_features)} climbs")
print(f"Feature columns: {len(df_features.columns)}")

print("\n### Feature Table Sample\n")
display(df_features.head(10))

In [ ]:
"""
==================================
Feature Summary Statistics
==================================
"""

print("### Feature Summary\n")

summary = df_features.describe().T
summary['missing'] = df_features.isna().sum()
summary['missing_pct'] = (df_features.isna().sum() / len(df_features) * 100).round(2)

display(summary[['count', 'mean', 'std', 'min', 'max', 'missing', 'missing_pct']])

## Correlation with Difficulty

In [ ]:
"""
==================================
Correlation with Difficulty
==================================
"""

correlations = df_features.corr()['display_difficulty'].drop('display_difficulty').sort_values(key=abs, ascending=False)

print("### Top 30 Features Correlated with Difficulty\n")
display(correlations.head(30).to_frame('correlation'))

print("\n### Bottom 10 Features (Least Correlated)\n")
display(correlations.tail(10).to_frame('correlation'))

# Visualizing Key Features

In [ ]:
"""
==================================
Visualize Key Features
==================================
"""

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

key_features = [
    # Core driver
    'angle',

    # Basic structure
    'total_holds',
    'height_gained',

    # Density / compactness
    'hold_density',
    'holds_per_vertical_foot',

    # Hand geometry (very important)
    'mean_hand_reach',
    'max_hand_reach',
    'std_hand_reach',

    # Hand-foot interaction
    'mean_hand_to_foot',
    'std_hand_to_foot',

    # Spatial layout
    'symmetry_score',
    'upper_ratio',

    # Global geometry
    'convex_hull_area',
    'hull_area_to_bbox_ratio',

    # Path / flow
    'path_length_vertical',
    'path_efficiency'
]

for ax, feature in zip(axes.flat, key_features):
    if feature in df_features.columns:
        ax.scatter(df_features[feature], df_features['display_difficulty'], alpha=0.3, s=10)
        ax.set_xlabel(feature)
        ax.set_ylabel('Difficulty')
        ax.set_title(f'{feature} vs Difficulty')

plt.tight_layout()
plt.savefig('../images/04_climb_features/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
"""
==================================
Add Interaction Features
==================================
"""

# Angle interactions
df_features['angle_x_holds'] = df_features['angle'] * df_features['total_holds']
df_features['angle_squared'] = df_features['angle'] ** 2


# Complexity features
df_features['complexity_score'] = (
    df_features['total_holds'] * 
    df_features['mean_hand_reach'].fillna(0) * 
    df_features['hold_density']
)

print(f"Added interaction features. Total columns: {len(df_features.columns)}")

In [ ]:
"""
==================================
Handle Missing Values
==================================
"""

missing = df_features.isna().sum()
missing_cols = missing[missing > 0]

print("### Columns with Missing Values\n")
display(missing_cols.to_frame('missing'))


# Fill other NaNs with column means
for col in df_features.columns:
    if df_features[col].isna().any():
        if df_features[col].dtype in ['float64', 'int64']:
            df_features[col] = df_features[col].fillna(df_features[col].mean())

# Check remaining missing
remaining = df_features.isna().sum().sum()
print(f"\nRemaining missing values: {remaining}")

In [ ]:
"""
===================================
Feature Importance Review
===================================
"""

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

X = df_features.drop(columns=['display_difficulty'])
y = df_features['display_difficulty']

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=3, n_jobs=-1)
rf.fit(X, y)

importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("### Top 30 Most Important Features (Random Forest)\n")
display(importance.head(30))

# Cross-validation score
scores = cross_val_score(rf, X, y, cv=5, scoring='neg_mean_absolute_error')
print(f"\nCross-validated MAE: {-scores.mean():.2f} (+/- {scores.std():.2f})")

# Conclusion

In [ ]:
"""
============================
Save Feature Matrix
============================
"""
raw_cols = [c for c in df_features.columns if c.endswith('_raw')]
if raw_cols:
    print("Dropping raw columns from final climb feature matrix:")
    print(raw_cols)
    df_features = df_features.drop(columns=raw_cols)

# `climb_features.csv` is the canonical name used by later notebooks.
df_features.to_csv('../data/04_climb_features/climb_features.csv')

print("Saved feature matrix to:")
print("  - ../data/04_climb_features/climb_features.csv")

with open('../data/04_climb_features/feature_list.txt', 'w') as f:
    for col in df_features.columns:
        f.write(f"{col}\n")

print("\nFeature list saved to ../data/04_climb_features/feature_list.txt")


In [ ]:
"""
==================================
Final Feature Summary
==================================
"""

print("### Feature Engineering Complete\n")
print(f"Total climbs: {len(df_features)}")
print(f"Total features: {df_features.shape[1] - 1}")  # Exclude target
print(f"Target: display_difficulty")
print(f"Feature matrix shape: {df_features.shape}")

print("""\nInterpretation:
- Each row is a climb-angle observation.
- The target is `display_difficulty`.
- The predictors combine geometry and structure
- The next notebook tests how much predictive signal these engineered features actually contain.
""")
